# Notebook 12 — Model Comparison, Diebold–Mariano and MCS

## Purpose

This notebook answers two different questions:

1. **Which model has the smallest error?**
   - RMSE
   - MAE
   - directional accuracy

2. **Are the differences statistically meaningful?**
   - Diebold–Mariano test
   - Model Confidence Set

A smaller RMSE is descriptive. It does not automatically prove that one model is genuinely better.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Find the project root whether the notebook is launched from:
#   project/
# or:
#   project/notebooks/
CURRENT = Path.cwd().resolve()

if (CURRENT / "data").exists():
    PROJECT_ROOT = CURRENT
elif CURRENT.name == "notebooks" and (CURRENT.parent / "data").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    possible_roots = [CURRENT, *CURRENT.parents]
    matches = [p for p in possible_roots if (p / "data").exists() and (p / "notebooks").exists()]
    if not matches:
        raise FileNotFoundError(
            "Could not find the project root. Open the sp500-forecasting-dissertation "
            "folder in VS Code, then run this notebook again."
        )
    PROJECT_ROOT = matches[0]

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORT_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURES = PROJECT_ROOT / "reports" / "figures"
MODEL_DIR = PROJECT_ROOT / "reports" / "models"

for folder in [DATA_PROCESSED, REPORT_TABLES, REPORT_FIGURES, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Python version:", sys.version.split()[0])

Project root: /home/claude/project/sp500-forecasting-dissertation
Python: /usr/bin/python3
Python version: 3.12.3


## Load prediction files

The expected minimum columns are:

- `date`
- `permno`
- `model`
- `actual`
- `forecast`

The notebook searches the processed-data folder for prediction parquet files created by earlier notebooks.

In [2]:
prediction_paths = sorted(
    p for p in DATA_PROCESSED.glob("*predictions*.parquet")
    if p.name != "all_model_predictions.parquet"
)

if not prediction_paths:
    raise FileNotFoundError(
        "No prediction parquet files were found. Run the walk-forward, "
        "LSTM/GRU and TFT notebooks first."
    )

frames = []

for path in prediction_paths:
    frame = pd.read_parquet(path).copy()
    frame.columns = [str(c).lower() for c in frame.columns]

    required = {"date", "model", "actual", "forecast"}

    if not required.issubset(frame.columns):
        print("Skipped because columns are missing:", path.name)
        continue

    if "permno" not in frame.columns:
        if "permno_string" in frame.columns:
            frame["permno"] = pd.to_numeric(
                frame["permno_string"],
                errors="coerce",
            )
        else:
            print("Skipped because no stock identifier exists:", path.name)
            continue

    frame["source_file"] = path.name
    frames.append(
        frame[
            ["date", "permno", "model", "actual", "forecast", "source_file"]
        ]
    )

if not frames:
    raise ValueError("Prediction files were found, but none had the required schema.")

predictions = pd.concat(frames, ignore_index=True)
predictions["date"] = pd.to_datetime(predictions["date"])
predictions["permno"] = pd.to_numeric(predictions["permno"], errors="coerce")
predictions["actual"] = pd.to_numeric(predictions["actual"], errors="coerce")
predictions["forecast"] = pd.to_numeric(predictions["forecast"], errors="coerce")

predictions = predictions.dropna(
    subset=["date", "permno", "model", "actual", "forecast"]
)

# Keep one forecast per date-stock-model.
# If the same model was saved more than once, the newest loaded row is retained.
predictions = (
    predictions.sort_values(["date", "permno", "model", "source_file"])
               .drop_duplicates(["date", "permno", "model"], keep="last")
               .reset_index(drop=True)
)

print("Files used:")
for path in prediction_paths:
    print("-", path.name)

print("\nModels:", sorted(predictions["model"].unique()))
print("Rows:", len(predictions))

Files used:
- lstm_gru_predictions.parquet
- tft_predictions.parquet
- walk_forward_classical_predictions.parquet

Models: ['ar1_garch_t', 'arma', 'gru', 'lstm', 'mean_baseline', 'tft']
Rows: 75480


## Align models fairly

A model comparison is only fair when models are evaluated on the same stock-date observations.

The next cell keeps the intersection shared by all selected models.

In [3]:
model_names = sorted(predictions["model"].unique())

model_counts_per_key = (
    predictions.groupby(["date", "permno"])["model"].nunique()
)

common_keys = (
    model_counts_per_key[
        model_counts_per_key == len(model_names)
    ]
    .reset_index()[["date", "permno"]]
)

aligned = predictions.merge(
    common_keys,
    on=["date", "permno"],
    how="inner",
)

print("Models compared:", model_names)
print("Common stock-date observations:", len(common_keys))
print("Aligned prediction rows:", len(aligned))

if common_keys.empty:
    raise ValueError(
        "No common stock-date observations exist across every model. "
        "This usually means the quick-mode notebooks used different stocks or dates. "
        "Run the models with a common universe and test period, or compare a selected "
        "subset of models first."
    )


Models compared: ['ar1_garch_t', 'arma', 'gru', 'lstm', 'mean_baseline', 'tft']
Common stock-date observations: 12580
Aligned prediction rows: 75480


## Descriptive metrics

- RMSE gives extra punishment to large errors.
- MAE gives the average absolute error.
- Directional accuracy checks the predicted sign.

Lower RMSE and MAE are better. Higher directional accuracy is better.

In [4]:
def calculate_metrics(group):
    error = group["actual"] - group["forecast"]

    return pd.Series({
        "observations": len(group),
        "rmse": np.sqrt(np.mean(error ** 2)),
        "mae": np.mean(np.abs(error)),
        "directional_accuracy": np.mean(
            np.sign(group["actual"]) == np.sign(group["forecast"])
        ),
    })


metrics_table = (
    aligned.groupby("model")
           .apply(calculate_metrics)
           .reset_index()
           .sort_values("rmse")
)

metrics_table

,model,observations,rmse,mae,directional_accuracy
2,gru,12580.0,0.020281,0.013772,0.527027
3,lstm,12580.0,0.020297,0.013779,0.524881
0,ar1_garch_t,12580.0,0.020350,0.013787,0.527424
1,arma,12580.0,0.020354,0.013790,0.526709
4,mean_baseline,12580.0,0.020355,0.013792,0.527981
5,tft,12580.0,0.020376,0.013817,0.525596


## Daily cross-sectional losses

Stocks on the same day react to common market events, so treating every stock-day as fully independent is unrealistic.

For each date and model:

1. calculate each stock's loss;
2. average losses across stocks;
3. compare the resulting daily loss series through time.

In [5]:
aligned = aligned.copy()
aligned["squared_error"] = (
    aligned["actual"] - aligned["forecast"]
) ** 2

aligned["absolute_error"] = (
    aligned["actual"] - aligned["forecast"]
).abs()

daily_mse = (
    aligned.groupby(["date", "model"])["squared_error"]
           .mean()
           .unstack("model")
           .dropna()
)

daily_mae = (
    aligned.groupby(["date", "model"])["absolute_error"]
           .mean()
           .unstack("model")
           .dropna()
)

print("Daily MSE matrix shape:", daily_mse.shape)
daily_mse.head()

Daily MSE matrix shape: (1258, 6)


model,ar1_garch_t,arma,gru,lstm,mean_baseline,tft
date,,,,,,
2020-01-02,0.000321,0.000329,0.000321,0.000337,0.000327,0.000356
2020-01-03,0.000113,0.000109,0.000107,0.000099,0.000109,0.000077
2020-01-06,0.000183,0.000187,0.000186,0.000195,0.000187,0.000222
2020-01-07,0.000060,0.000059,0.000059,0.000055,0.000059,0.000053
2020-01-08,0.000097,0.000099,0.000100,0.000104,0.000099,0.000101


## Diebold–Mariano-style test with HAC standard errors

**Question:** Do two models have equal predictive accuracy?

**Null hypothesis:** Mean loss difference equals zero.

We estimate the mean loss difference with Newey–West/HAC standard errors so mild serial dependence does not make the result look more certain than it is.

Sign convention:

- negative mean difference: candidate has lower loss than benchmark;
- positive mean difference: candidate has higher loss than benchmark.

In [6]:
import statsmodels.api as sm


def dm_hac_test(
    candidate_losses: pd.Series,
    benchmark_losses: pd.Series,
    max_lags: int = 5,
):
    combined = pd.concat(
        [candidate_losses, benchmark_losses],
        axis=1,
    ).dropna()

    differential = combined.iloc[:, 0] - combined.iloc[:, 1]

    if len(differential) < 20:
        return {
            "mean_loss_difference": np.nan,
            "test_statistic": np.nan,
            "p_value": np.nan,
            "observations": len(differential),
        }

    design = np.ones((len(differential), 1))

    fitted = sm.OLS(
        differential.to_numpy(),
        design,
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": max_lags},
    )

    return {
        "mean_loss_difference": float(differential.mean()),
        "test_statistic": float(fitted.tvalues[0]),
        "p_value": float(fitted.pvalues[0]),
        "observations": len(differential),
    }


preferred_benchmarks = [
    "mean_baseline",
    "random_walk",
    "arma",
]

benchmark_model = next(
    (name for name in preferred_benchmarks if name in daily_mse.columns),
    daily_mse.columns[0],
)

dm_rows = []

for candidate in daily_mse.columns:
    if candidate == benchmark_model:
        continue

    result = dm_hac_test(
        daily_mse[candidate],
        daily_mse[benchmark_model],
        max_lags=5,
    )

    dm_rows.append({
        "candidate": candidate,
        "benchmark": benchmark_model,
        **result,
        "significant_at_5pct": (
            result["p_value"] < 0.05
            if pd.notna(result["p_value"])
            else False
        ),
    })

dm_table = pd.DataFrame(dm_rows).sort_values("p_value")
dm_table

,candidate,benchmark,mean_loss_difference,test_statistic,p_value,observations,significant_at_5pct
2,gru,mean_baseline,-3.025304e-06,-1.729404,0.083737,1258,False
3,lstm,mean_baseline,-2.340595e-06,-1.642681,0.100449,1258,False
0,ar1_garch_t,mean_baseline,-2.061820e-07,-0.932457,0.351101,1258,False
4,tft,mean_baseline,8.750142e-07,0.921245,0.356922,1258,False
1,arma,mean_baseline,-5.137358e-08,-0.244865,0.806561,1258,False


## Model Confidence Set

**Question:** When several models are compared together, which models cannot be statistically separated from the best?

The MCS starts with all models and removes clearly inferior ones.

The planned dissertation confidence level is 90%, so `size=0.10`.

In [7]:
from arch.bootstrap import MCS

MCS_REPETITIONS = 5000
MCS_SIZE = 0.10

mcs = MCS(
    daily_mse,
    size=MCS_SIZE,
    reps=MCS_REPETITIONS,
    block_size=10,
    method="R",
    seed=42,
)

mcs.compute()

print("Models included in the confidence set:")
print(mcs.included)

mcs_pvalues = mcs.pvalues
mcs_pvalues

Models included in the confidence set:
['ar1_garch_t', 'arma', 'gru', 'lstm', 'mean_baseline', 'tft']


,Pvalue
Model name,
tft,0.1720
arma,0.3268
mean_baseline,0.3268
ar1_garch_t,0.3268
lstm,0.3268
gru,1.0000


In [8]:
metrics_table.to_csv(
    REPORT_TABLES / "all_model_metrics.csv",
    index=False,
)

dm_table.to_csv(
    REPORT_TABLES / "diebold_mariano_hac_results.csv",
    index=False,
)

mcs_pvalues.to_csv(
    REPORT_TABLES / "model_confidence_set_pvalues.csv"
)

aligned.to_parquet(
    DATA_PROCESSED / "all_model_predictions.parquet",
    index=False,
)

print("Saved model-comparison outputs.")

Saved model-comparison outputs.


## What to say in the meeting

> RMSE and MAE tell me which numerical error is smaller. The Diebold–Mariano test asks whether the loss difference against a benchmark is statistically different from zero. MCS compares all models together and keeps the group that cannot be statistically distinguished from the best. This prevents me from declaring a winner based only on a tiny difference in RMSE.